# Занятие 9, пара 2. Экономика проекта с данными


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## Модель бесплатная. А проект?

В предыдущей паре мы отделяли вычисления от качества. Теперь отделим цену одной операции от стоимости
всего проекта. Даже если библиотека бесплатная, данные надо получить, разметить, проверить,
а работающий результат — поддерживать.

Соберём небольшой расчёт по шагам, затем проверим, какие предположения сильнее всего меняют решение.
Все суммы ниже — **вымышленные учебные условия в условных денежных единицах (у.е.)**, не рыночные тарифы.
Здесь не нужны регистрация, облако и платные API. Итог — таблица бюджета, сравнение сценариев и короткое
обоснование следующего шага своего проекта.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check

PROJECT_DIR


### 1. Сначала ограничим задачу

Представим сервис, который сортирует обращения по темам. На пилоте нужно разметить 1200 обращений,
проверить качество и провести вычислительный эксперимент. Считаем **первые три месяца** проекта.
Сравнивать стоимость трёх месяцев облака с годом работы сервера бессмысленно, даже если формулы верны.

Зафиксируем границы: учитываем труд по заданным учебным ставкам, подготовку, проверку, вычисления,
хранение и поддержку. Налоги, закупочные процедуры и стоимость капитала не моделируем.
Честный расчёт должен называть исключения, а не выдавать себя за полный коммерческий бизнес-план.

Начнём с одной размеченной записи. 24 секунды — это 24/3600 часа, а не 0.24 часа.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

seconds_per_label = 24
hourly_rate = 600
one_label_hours = seconds_per_label / 3600
one_label_cost = one_label_hours * hourly_rate
print('Часов на одно решение:', one_label_hours)
print('Стоимость решения, у.е.:', one_label_cost)


### 2. Объектов и решений может быть разное количество

Чтобы проверить согласованность, четверть обращений дадим второму человеку независимо.
Каждое обращение получает первое решение; выбранная четверть — ещё одно.
Итого `N + N × overlap_share`, а не `N × 2` для всего набора.
Доля 0.25 означает 25%, не 25. Для исходных условий число повторов целое; в планировании произвольных
объёмов число оплачиваемых решений округляют вверх и явно описывают правило.


In [ ]:
n_objects = 1200
overlap_share = 0.25
n_decisions = n_objects + n_objects * overlap_share
annotation_hours = n_decisions * seconds_per_label / 3600
annotation_cost = annotation_hours * hourly_rate
print('Объектов:', n_objects, 'решений:', n_decisions)
print('Человеко-часов:', annotation_hours, 'стоимость:', annotation_cost)


Десять человеко-часов не гарантируют завершение за десять часов по календарю.
Есть очередь, перерывы, согласование инструкции и общая проверка. Пока считаем объём труда.
Повторная разметка — не то же самое, что исправление брака: это плановая проверка качества.

Если часть решений пришлось переделать, её труд прибавляется отдельно. В следующем упражнении
для простоты у каждого объекта одна первоначальная попытка и не более одной переделки.


### ✏️ Ваш ход — Самая быстрая попытка не всегда выгоднее

Два вымышленных процесса на 1000 объектах при одинаковой ставке:
A: 15 секунд на первый проход, у 40% нужна переделка по 60 секунд.
B: 25 секунд на первый проход, у 10% нужна переделка по 30 секунд.

Посчитайте `seconds_a`, `seconds_b` — суммарные секунды труда, выберите процесс и объясните решение.
Условие предполагает, что после переделки качество одинаковое. Как изменится рассуждение,
если это предположение не подтвердится?


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('09.2.1', seconds_a, seconds_b)


**Мой вывод:** …


### 3. Добавим то, что не попало в цену разметки

Из повторно размеченной части ожидаем 60 спорных случаев: эксперт тратит на каждый 120 секунд,
его ставка 900 у.е./час. Ещё 4 часа нужны для подготовки инструкции и примеров.
Вычисления — 8 часов по 50 у.е.; хранение — 200 у.е. в месяц.
Поддержка — 2 часа в месяц по ставке эксперта. Это входные **допущения**, не измеренные факты.


In [ ]:
months = 3
expert_rate = 900
review_cost = 60 * 120 / 3600 * expert_rate
preparation_cost = 4 * expert_rate
compute_cost = 8 * 50
storage_cost = months * 200
maintenance_cost = months * 2 * expert_rate

budget = pd.DataFrame([
    {'item': 'Подготовка', 'cost': preparation_cost},
    {'item': 'Разметка', 'cost': annotation_cost},
    {'item': 'Разбор разногласий', 'cost': review_cost},
    {'item': 'Вычисления', 'cost': compute_cost},
    {'item': 'Хранение', 'cost': storage_cost},
    {'item': 'Поддержка', 'cost': maintenance_cost},
])
display(budget)
base_total = budget['cost'].sum()
print('Итого за 3 месяца, у.е.:', base_total)


Список словарей стал таблицей; сумма cost — общий бюджет без резерва.
Посмотрим состав, чтобы не оптимизировать самую маленькую статью просто потому, что её проще менять.
`barh` — горизонтальные столбцы из 08.1, удобные для длинных названий.


In [ ]:
ordered = budget.sort_values('cost')
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(ordered['item'], ordered['cost'])
ax.set_xlabel('Условные денежные единицы за 3 месяца')
ax.set_title('Учебный бюджет: что вносит основной вклад?')
plt.show()


Резерв на непредвиденное не заменяет учёт известных работ. Например, нельзя убрать поддержку
из таблицы и надеяться, что её покроет случайная надбавка.
Для учебного сценария добавим 15% **поверх всех учтённых затрат**. Это принятое нами правило,
не обязательный отраслевой норматив и не статистическая вероятность уложиться в сумму.


In [ ]:
reserve_share = 0.15
reserve_cost = base_total * reserve_share
planned_total = base_total + reserve_cost
print('Без резерва:', base_total)
print('Резерв:', reserve_cost)
print('Плановая сумма:', planned_total)


### 4. Какие числа опаснее всего угадать неправильно?

Время на запись мы придумали. Проверим три сценария: 12, 24 и 48 секунд.
Все остальные допущения оставляем одинаковыми. Это **анализ чувствительности**:
меняем один вход и смотрим, насколько изменился результат.
Функция ниже собирает уже разобранные формулы; она не предсказывает реальную стоимость автоматически.
Число спорных случаев в этой версии фиксировано: при изменении объёма его тоже пришлось бы пересчитать.


In [ ]:
def estimate_budget(label_seconds, n=1200, overlap=0.25):
    decisions = n * (1 + overlap)
    labeling = decisions * label_seconds / 3600 * 600
    fixed = 4 * 900 + 60 * 120 / 3600 * 900 + 8 * 50 + 3 * 200 + 3 * 2 * 900
    subtotal = labeling + fixed
    return {'label_seconds': label_seconds, 'annotation': labeling,
            'subtotal': subtotal, 'with_reserve': subtotal * 1.15}

scenarios = []
for seconds in [12, 24, 48]:
    scenarios.append(estimate_budget(seconds))
scenario_table = pd.DataFrame(scenarios)
display(scenario_table)


Из таблицы следует полезное действие: сначала измерить скорость на небольшом разнообразном пилоте.
Если взять только лёгкие записи, среднее время будет слишком оптимистичным.
Зафиксируйте также разногласия и переделки: ускорение ценой качества может вернуть расходы позже.

### 5. Облако или локальный ресурс: сначала одинаковая работа

Рассмотрим отдельную вычислительную часть на тот же период, три месяца.
Вымышленные условия: облако — 80 у.е. за полезный час; локальный ресурс — фиксированные 24000 у.е.
за период плюс 20 у.е. за полезный час. Фиксированная сумма — **уже выделенная доля стоимости на период**,
не цена покупки компьютера. Сравниваем одинаковую выполненную работу и требуемую производительность.

При h часах получаем две прямые: `80h` и `24000+20h`.
Это упрощённый расчёт: доступность, настройка, ремонт, хранение, передача данных и ограничения мощности
надо включать отдельно при настоящем выборе. Не делаем покупку на основании учебных чисел.


In [ ]:
hours = [0, 100, 200, 300, 400, 500, 600]
cloud_costs = []
local_costs = []
for h in hours:
    cloud_costs.append(80 * h)
    local_costs.append(24000 + 20 * h)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(hours, cloud_costs, marker='o', label='Облако')
ax.plot(hours, local_costs, marker='o', label='Локальный ресурс')
ax.set_xlabel('Полезные часы за 3 месяца')
ax.set_ylabel('Условные денежные единицы')
ax.legend()
ax.set_title('Сравнение только при перечисленных допущениях')
plt.show()


Точка равенства: `cloud_rate × h = fixed + local_rate × h`.
Переносим переменные: `(cloud_rate−local_rate) × h = fixed`, значит `h = fixed / разницу ставок`.
При положительном fixed и облачной ставке не выше локальной положительной точки равенства нет:
локальный вариант в этой модели не становится дешевле с ростом нагрузки.


### ✏️ Ваш ход — Точка равенства ещё не решение

Напишите `break_even(fixed, cloud_rate, local_rate)` для положительного fixed и неотрицательных ставок.
Верните часы равенства, а если положительного пересечения нет — None.
Проверьте исходные ставки и случай 24000, 20, 30.
Даже если получили 400 часов, можно ли выбрать локальный вариант, когда ресурс доступен только 150 часов за период?


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('09.2.2', break_even)


**Мой вывод:** …


### 6. Человеко-часы и календарные часы

Если работа требует 12 человеко-часов, двое людей могут в идеальном случае выполнить независимую часть
за 6 часов. Но трудозатраты останутся 12 человеко-часов. Общую подготовку и проверку нельзя всегда распараллелить.

Маленький пример: 2 часа общей подготовки, 12 часов независимой разметки, 1 час общей проверки.
Один исполнитель — 15 часов. Двое — в идеальной модели 2+12/2+1=9 часов.
Это нижняя оценка календарного времени без очередей и координации.


In [ ]:
prep_hours = 2
parallel_work = 12
review_hours = 1
workers = 2
calendar_lower_bound = prep_hours + parallel_work / workers + review_hours
labor_hours = prep_hours + parallel_work + review_hours
print('Календарные часы не меньше:', calendar_lower_bound)
print('Человеко-часы:', labor_hours)


### ✏️ Ваш ход — Нанять ещё людей — сэкономить?

Для другой задачи нужны 3 часа общей подготовки, 24 человеко-часа независимой работы
и 2 часа общей проверки. Все стадии идут последовательно; подготовку и проверку делает один человек,
а независимую часть делят четверо. Ставка у всех 500 у.е./час.
Запишите `calendar_hours` и `labor_cost`. Почему расчёт `(3+24+2)/4` неверен?
Назовите одну причину, по которой реальный срок может оказаться больше вашей оценки.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('09.2.3', calendar_hours, labor_cost)


**Мой вывод:** …


## Решение для своего проекта

Сформулируйте один небольшой следующий шаг: провести пилот, уточнить инструкцию, уменьшить область задачи
или проверить доступность вычислений. Обоснуйте его одной строкой расчёта и одним ограничением.
Не надо придумывать коммерческую прибыль для учебного проекта.

Следующая ячейка сохраняет основной расчёт; ответы упражнений ей не нужны.


In [ ]:
output_dir = WORKSPACE_DIR / 'outputs' / 'project_budget'
output_dir.mkdir(parents=True, exist_ok=True)
budget.to_csv(output_dir / 'budget.csv', index=False)
scenario_table.to_csv(output_dir / 'scenarios.csv', index=False)
print(output_dir)


**Резерв на 10–15 минут:** составьте для своего проекта три предположения — оптимистичное,
рабочее и осторожное. Попросите соседа найти забытый этап или несовместимые единицы.
Сценарии не заменяют измерения, но помогают выбрать, что измерить первым.

Сегодня не появляется новая ЛР. Расчёт можно сохранить как дополнительное объяснение ограничений проекта.
В следующем блоке разберём саму разметку: почему понятная инструкция и проверка разногласий влияют и на качество,
и на стоимость. Доступ к платным ресурсам не требуется.
